In [1]:
# 1) Instalar dependencias
!pip install -q chromadb sentence-transformers pypdf

# 2) Subir tus archivos locales de forma interactiva
from google.colab import files
from pathlib import Path

# Crear la carpeta de destino en Colab
pdf_dir = Path("/content/pdfs")
pdf_dir.mkdir(parents=True, exist_ok=True)

print("📂 Haz clic en el botón de abajo para seleccionar y subir tus dos PDFs desde tu computadora:")
subidos = files.upload()

# Mover los archivos subidos a la carpeta /content/pdfs
for nombre_archivo in subidos.keys():
    origen = Path(nombre_archivo)
    destino = pdf_dir / nombre_archivo
    origen.rename(destino)
    print(f"✅ Archivo guardado en Colab: {destino.name}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
📂 Haz clic en el botón de abajo para seleccionar y subir tus dos PDFs desde tu computadora:


Saving manual_rrhh.pdf to manual_rrhh.pdf
Saving manual_ti.pdf to manual_ti.pdf
✅ Archivo guardado en Colab: manual_rrhh.pdf
✅ Archivo guardado en Colab: manual_ti.pdf


In [2]:
# 3) Importar librerías restantes
import chromadb
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader

# Configuración de rutas y modelo
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
db_path = Path("/content/chroma_db")

# Extraer texto de PDF
def extraer_texto_pdf(archivo_pdf: Path) -> str:
    lector = PdfReader(str(archivo_pdf))
    textos = []
    for pagina_num, pagina in enumerate(lector.pages, start=1):
        texto = pagina.extract_text() or ""
        texto = " ".join(texto.split())
        if texto:
            textos.append(f"[Página {pagina_num}] {texto}")
    return "\n".join(textos)

# Dividir texto en chunks
def dividir_texto(texto: str, chunk_size: int = 500, overlap: int = 80):
    if not texto:
        return []
    chunks = []
    inicio = 0
    longitud = len(texto)

    while inicio < longitud:
        fin = min(inicio + chunk_size, longitud)
        chunk = texto[inicio:fin].strip()
        if chunk:
            chunks.append(chunk)
        if fin == longitud:
            break
        inicio += chunk_size - overlap

    return chunks

# Indexar PDFs
def indexar_pdfs(pdf_dir: Path, db_path: Path):
    modelo = SentenceTransformer(MODEL_NAME)

    cliente = chromadb.PersistentClient(path=str(db_path))
    coleccion = cliente.get_or_create_collection(
        name="manuales",
        metadata={"hnsw:space": "cosine"}
    )

    ids = []
    documentos = []
    metadatos = []

    archivos_pdf = sorted(pdf_dir.glob("*.pdf"))
    if not archivos_pdf:
        raise ValueError(f"⚠️ No se encontraron archivos PDF en: {pdf_dir}")

    for pdf_path in archivos_pdf:
        texto_completo = extraer_texto_pdf(pdf_path)
        chunks = dividir_texto(texto_completo)

        for idx, chunk in enumerate(chunks):
            chunk_id = f"{pdf_path.stem}_{idx:03d}"
            ids.append(chunk_id)
            documentos.append(chunk)
            metadatos.append({
                "fuente": pdf_path.name,
                "archivo": pdf_path.stem,
                "chunk": idx
            })

    embeddings = modelo.encode(documentos, normalize_embeddings=True).tolist()

    coleccion.upsert(
        ids=ids,
        documents=documentos,
        metadatas=metadatos,
        embeddings=embeddings
    )

    print(f"✅ Se indexaron {len(documentos)} chunks desde {len(archivos_pdf)} PDF(s).")
    return coleccion, modelo

# Función de Consulta
def consultar(coleccion, modelo, pregunta: str, n_results: int = 3, fuente: str = None):
    embedding = modelo.encode([pregunta], normalize_embeddings=True).tolist()

    if fuente:
        resultado = coleccion.query(
            query_embeddings=embedding,
            n_results=n_results,
            where={"fuente": fuente}
        )
    else:
        resultado = coleccion.query(
            query_embeddings=embedding,
            n_results=n_results
        )

    return resultado

# Ejecución principal
coleccion, modelo = indexar_pdfs(pdf_dir, db_path)

pregunta = "¿De qué tratan los documentos?"  # Cámbiala por una pregunta específica de tus textos
resultado = consultar(coleccion, modelo, pregunta, n_results=3)

print(f"\n🔍 Pregunta: {pregunta}")
for i, (doc, dist) in enumerate(zip(resultado["documents"][0], resultado["distances"][0]), start=1):
    fuente = resultado["metadatas"][0][i - 1].get("fuente", "desconocida")
    print(f"\n[{i}] Fuente: {fuente}")
    print(f"Texto: {doc}")
    print(f"Distancia coseno: {dist:.4f}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Se indexaron 88 chunks desde 2 PDF(s).

🔍 Pregunta: ¿De qué tratan los documentos?

[1] Fuente: manual_rrhh.pdf
Texto: dor durante los primeros 5 días de cada trimestre. Manual de Recursos Humanos v4.1 Página 11 de 15
[Página 12] 15. Protección de Datos Personales del Colaborador (Habeas Data) La organización garantiza el tratamiento conﬁdencial y la protección integral de los datos personales y sensibles almacenados en el expediente laboral. 15.1 Derechos ARCO (Acceso, Rectiﬁcación, Cancelación y Oposición) Derecho ARCO Descripción de la Solicitud Plazo de Respuesta RRHH Acceso Consultar los datos guardados en
Distancia coseno: 0.4968

[2] Fuente: manual_rrhh.pdf
Texto: . Aprobación Institucional de Políticas de RRHH Aprobado por: Vicepresidencia de Recursos Humanos, Dirección General y Asesoría Jurídica Laboral. Fecha de Publicación Oﬁcial: 1 de Enero de 2026. Sede Principal Corporativa. Manual de Recursos Humanos v4.1 Página 15 de 15
Distancia coseno: 0.5008

[3] Fuente: manual_ti

In [3]:
# ==========================================
# CELDA 2: Procesamiento, Indexación y Comparativa de Similitud
# ==========================================
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader

# Configuración de rutas y modelo
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
db_path = Path("/content/chroma_db")
pdf_dir = Path("/content/pdfs")

# 1) Extraer texto de PDF
def extraer_texto_pdf(archivo_pdf: Path) -> str:
    lector = PdfReader(str(archivo_pdf))
    textos = []
    for pagina_num, pagina in enumerate(lector.pages, start=1):
        texto = pagina.extract_text() or ""
        texto = " ".join(texto.split())
        if texto:
            textos.append(f"[Página {pagina_num}] {texto}")
    return "\n".join(textos)

# 2) Dividir texto en chunks
def dividir_texto(texto: str, chunk_size: int = 500, overlap: int = 80):
    if not texto:
        return []
    chunks = []
    inicio = 0
    longitud = len(texto)

    while inicio < longitud:
        fin = min(inicio + chunk_size, longitud)
        chunk = texto[inicio:fin].strip()
        if chunk:
            chunks.append(chunk)
        if fin == longitud:
            break
        inicio += chunk_size - overlap

    return chunks

# 3) Indexar PDFs
def indexar_pdfs(pdf_dir: Path, db_path: Path):
    modelo = SentenceTransformer(MODEL_NAME)

    cliente = chromadb.PersistentClient(path=str(db_path))
    coleccion = cliente.get_or_create_collection(
        name="manuales",
        metadata={"hnsw:space": "cosine"}
    )

    ids = []
    documentos = []
    metadatos = []

    archivos_pdf = sorted(pdf_dir.glob("*.pdf"))
    if not archivos_pdf:
        raise ValueError(f"⚠️ No se encontraron archivos PDF en: {pdf_dir}")

    for pdf_path in archivos_pdf:
        texto_completo = extraer_texto_pdf(pdf_path)
        chunks = dividir_texto(texto_completo)

        for idx, chunk in enumerate(chunks):
            chunk_id = f"{pdf_path.stem}_{idx:03d}"
            ids.append(chunk_id)
            documentos.append(chunk)
            metadatos.append({
                "fuente": pdf_path.name,
                "archivo": pdf_path.stem,
                "chunk": idx
            })

    embeddings = modelo.encode(documentos, normalize_embeddings=True).tolist()

    coleccion.upsert(
        ids=ids,
        documents=documentos,
        metadatas=metadatos,
        embeddings=embeddings
    )

    print(f"✅ Se indexaron {len(documentos)} chunks desde {len(archivos_pdf)} PDF(s).\n")
    return coleccion, modelo, archivos_pdf

# ==========================================
# Ejecución e インデックス (Indexación)
# ==========================================
coleccion, modelo, archivos_pdf = indexar_pdfs(pdf_dir, db_path)

# Frases de prueba solicitadas para la comparativa
frases_prueba = [
    "Para restablecer la clave, ingrese a Configuración...",
    "Las vacaciones se solicitan con 15 días de anticipación..."
]

# Total de chunks en la base de datos para dimensionar la consulta global
total_chunks = len(coleccion.get()["ids"])

print("=" * 70)
print("📊 TABLA COMPARATIVA DE SIMILITUD COSENE POR CADA PDF")
print("=" * 70)

for frase in frases_prueba:
    print(f"\n🔍 Frase evaluada: \"{frase}\"")
    print("-" * 70)

    # Vectorizar la frase
    vector_frase = modelo.encode([frase], normalize_embeddings=True).tolist()

    # Consultar todos los documentos para evaluar contra cada PDF
    resultado = coleccion.query(
        query_embeddings=vector_frase,
        n_results=total_chunks
    )

    # Encontrar la máxima similitud para cada PDF fuente
    mejores_por_fuente = {}
    for doc, dist, meta in zip(resultado["documents"][0], resultado["distances"][0], resultado["metadatas"][0]):
        fuente = meta.get("fuente", "desconocida")
        # Similitud coseno = 1 - distancia coseno (a mayor valor, mayor parecido)
        similitud = 1 - dist

        if fuente not in mejores_por_fuente or similitud > mejores_por_fuente[fuente]["similitud"]:
            mejores_por_fuente[fuente] = {
                "similitud": similitud,
                "distancia": dist,
                "fragmento": doc
            }

    # Mostrar resultados por cada PDF disponible en la carpeta
    for pdf_path in archivos_pdf:
        nombre_pdf = pdf_path.name
        if nombre_pdf in mejores_por_fuente:
            data = mejores_por_fuente[nombre_pdf]
            print(f"  📄 Archivo: {nombre_pdf}")
            print(f"     🔹 Similitud Coseno : {data['similitud']:.4f}  (Distancia: {data['distancia']:.4f})")
            print(f"     💬 Fragmento clave  : \"{data['fragmento'][:120]}...\"")
        else:
            print(f"  📄 Archivo: {nombre_pdf}")
            print(f"     ⚠️ Sin coincidencias relevantes indexadas.")
        print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Se indexaron 88 chunks desde 2 PDF(s).

📊 TABLA COMPARATIVA DE SIMILITUD COSENE POR CADA PDF

🔍 Frase evaluada: "Para restablecer la clave, ingrese a Configuración..."
----------------------------------------------------------------------
  📄 Archivo: manual_rrhh.pdf
     🔹 Similitud Coseno : 0.3632  (Distancia: 0.6368)
     💬 Fragmento clave  : "porte inmediato a Servicios Generales Costo de reposición: $10 USD El carnet de identiﬁcación contiene un chip de proxim..."

  📄 Archivo: manual_ti.pdf
     🔹 Similitud Coseno : 0.5292  (Distancia: 0.4708)
     💬 Fragmento clave  : "dar: Restablecimiento de Credenciales Para restablecer su clave institucional de acceso a la red y VPN, ingrese al porta..."


🔍 Frase evaluada: "Las vacaciones se solicitan con 15 días de anticipación..."
----------------------------------------------------------------------
  📄 Archivo: manual_rrhh.pdf
     🔹 Similitud Coseno : 0.6802  (Distancia: 0.3198)
     💬 Fragmento clave  : "es. 1. Solicitud y Goce de V